In [ ]:
from langfuse import Langfuse
# from langfuse.openai import AzureOpenAI
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage
import operator
from statistics import mode
import os



langfuse = Langfuse(
    public_key=os.getenv("LANGFUSE_PUBLIC_KEY"),
    secret_key=os.getenv("LANGFUSE_SECRET_KEY"),
    host=os.getenv("LANGFUSE_HOST")
)

# Define calculator tools
@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract the second number from the first."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide the first number by the second."""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# Define tools list and create agent
tools = [add, subtract, multiply, divide]

# Prompts for the calculator
expressions_prompt = """Generate a mathematical expression that needs to be calculated. Single operator expression. Use basic operations (+,-,*,/). Example: '40 / 5' or '12 * 3'"""
calculate_prompt = """Calculate this expression: {expression}. Only use the available tools to execute the expressions and return only the numerical result."""

load_dotenv()
# LLM
model = AzureChatOpenAI(model="gpt-4o-mini", api_version="2024-08-01-preview", temperature=1)

# Create calculator agent
calculator_prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="You are a helpful calculator assistant that uses tools to solve math problems."),
    HumanMessagePromptTemplate.from_template("{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_openai_tools_agent(model,tools, calculator_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, return_intermediate_steps=True)

In [82]:
# Second cell - Update model classes
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel
# from langgraph.constants import Send
from langgraph.types import Send
from langgraph.graph import END, StateGraph, START
from statistics import mode

class Expression(BaseModel):
    expression: str

class CalculationState(TypedDict):
    expression: str
    execution_id: int

class Result(BaseModel):
    result: float

# Third cell - Update state class
class OverallState(TypedDict):
    expression: str
    results: Annotated[list, operator.add]
    final_result: float

In [92]:
agent_executor.invoke({
        "input": f"Calculate this expression: '2-5'. Return only the numerical result. Don't include any text"
    })

{'input': "Calculate this expression: '2-5'. Return only the numerical result. Don't include any text",
 'output': '-3.0',
 'intermediate_steps': [(ToolAgentAction(tool='subtract', tool_input={'a': 2, 'b': 5}, log="\nInvoking: `subtract` with `{'a': 2, 'b': 5}`\n\n\n", message_log=[AIMessageChunk(content='', additional_kwargs={'tool_calls': [{'index': 0, 'id': 'call_loC9sKbvibxXtGnaSjc6GCtZ', 'function': {'arguments': '{"a":2,"b":5}', 'name': 'subtract'}, 'type': 'function'}]}, response_metadata={'finish_reason': 'tool_calls', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b'}, id='run--14fc895c-561d-4362-8032-2e1d7672aa48', tool_calls=[{'name': 'subtract', 'args': {'a': 2, 'b': 5}, 'id': 'call_loC9sKbvibxXtGnaSjc6GCtZ', 'type': 'tool_call'}], tool_call_chunks=[{'name': 'subtract', 'args': '{"a":2,"b":5}', 'id': 'call_loC9sKbvibxXtGnaSjc6GCtZ', 'index': 0, 'type': 'tool_call_chunk'}])], tool_call_id='call_loC9sKbvibxXtGnaSjc6GCtZ'),
   -3.0)]}

In [83]:
from langfuse import observe

# Fourth cell - Update processing functions
@observe
def generate_expression(state: OverallState):
    response = model.with_structured_output(Expression).invoke(expressions_prompt)
    return {"expression": response.expression}

@observe
def create_parallel_executions(state: OverallState):
    # Create 3 parallel executions of the same expression
    return [Send("calculate", {"expression": state["expression"], "execution_id": i}) for i in range(3)]

@observe
def calculate(state: CalculationState):
    response = agent_executor.invoke({
        "input": f"Calculate this expression: {state['expression']}. Return only the numerical result."
    })
    try:
        result = float(response["output"])
        print(f"Raw output: {response}")
    except ValueError:
        result = 0.0  # fallback in case of parsing errors
    print(f"Execution {state['execution_id']}: {state['expression']} = {result}")
    return {"results": [result]}

@observe
def aggregate_results(state: OverallState):
    # Calculate mode of the results
    mode_value = mode(state["results"])
    print(f"\nAll results: {state['results']}")
    print(f"Mode value: {mode_value}")
    return {"final_result": mode_value}

# Create graph
graph = StateGraph(OverallState)
graph.add_node("generate_expression", generate_expression)
graph.add_node("calculate", calculate)
graph.add_node("aggregate_results", aggregate_results)
graph.add_edge(START, "generate_expression")
graph.add_conditional_edges("generate_expression", create_parallel_executions, ["calculate"])
graph.add_edge("calculate", "aggregate_results")
graph.add_edge("aggregate_results", END)

# Compile the graph
app = graph.compile()

In [ ]:
# Fifth cell - Test the parallel calculator
for s in app.stream({}):
    print(s)

{'generate_expression': {'expression': '15 + 27'}}
Raw output: {'input': 'Calculate this expression: 15 + 27. Return only the numerical result.', 'output': '42.0'}
Execution 1: 15 + 27 = 42.0
{'calculate': {'results': [42.0]}}
Raw output: {'input': 'Calculate this expression: 15 + 27. Return only the numerical result.', 'output': '42'}
Execution 2: 15 + 27 = 42.0
{'calculate': {'results': [42.0]}}
Raw output: {'input': 'Calculate this expression: 15 + 27. Return only the numerical result.', 'output': '42.0'}
Execution 0: 15 + 27 = 42.0
{'calculate': {'results': [42.0]}}

All results: [42.0, 42.0, 42.0]
Mode value: 42.0
{'aggregate_results': {'final_result': 42.0}}


In [85]:
print(app.get_graph().draw_ascii())

     +-----------+       
     | __start__ |       
     +-----------+       
            *            
            *            
            *            
+---------------------+  
| generate_expression |  
+---------------------+  
            .            
            .            
            .            
     +-----------+       
     | calculate |       
     +-----------+       
            *            
            *            
            *            
 +-------------------+   
 | aggregate_results |   
 +-------------------+   
            *            
            *            
            *            
      +---------+        
      | __end__ |        
      +---------+        
